# Aave V3.1 — feature addition

Loads the **transformed** frames from `transformed_data/`:

- `DF_common_1` — asset-level reserve rates/indexes, keyed on `(time_bucket, asset)`
- `DF_common_final` — protocol-level 6h time series, keyed on `time_bucket`

In [1]:
# Load transformed data from transformed_data/
import pandas as pd
from pathlib import Path
from IPython.display import display
import adv_validation as adv

DATA_DIR = Path("transformed_data")
PREVIEW_ROWS = 10

DF_common_1 = pd.read_csv(DATA_DIR / "DF_common_1.csv")          # asset-level (time_bucket, asset)
DF_common_final = pd.read_csv(DATA_DIR / "DF_common_final.csv")  # protocol-level 6h series


print(f" {DF_common_final.shape[0]} rows x {DF_common_final.shape[1]} cols")
display(DF_common_final.head(PREVIEW_ROWS))

 4380 rows x 46 cols


,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth,borrow_tx_count,...,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,sampled_user_count,account_data_call_count
0,2025-04-01 00:00:00.000 UTC,107,112,54,60,1.814934e+08,99340.849932,9.766776e+07,53458.667287,54.0,...,20.0,19.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-04-01 02:00:00.000 UTC,116,105,60,55,2.512122e+08,136862.261136,2.665668e+08,145227.537741,58.0,...,19.0,19.0,4.0,1.491521e+08,4.997479e+07,6.710961e+07,0.81,0.785,1.0,1.0
2,2025-04-01 04:00:00.000 UTC,72,100,38,54,1.096440e+08,59557.933326,1.329771e+08,72232.382778,48.0,...,7.0,8.0,1.0,1.506325e+08,5.192650e+07,6.631999e+07,0.81,0.785,1.0,1.0
3,2025-04-01 06:00:00.000 UTC,129,115,58,48,6.726555e+08,362808.719631,6.691233e+08,360903.546607,64.0,...,7.0,6.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-04-01 08:00:00.000 UTC,138,118,68,53,3.680422e+08,195833.482106,3.695117e+08,196615.376327,95.0,...,17.0,17.0,11.0,1.262487e+03,4.772150e+02,4.696503e+02,0.78,0.750,1.0,1.0
5,2025-04-01 10:00:00.000 UTC,127,111,77,59,2.532615e+08,135086.519080,2.552678e+08,136156.675648,72.0,...,12.0,11.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2025-04-01 12:00:00.000 UTC,127,122,60,62,4.748889e+08,254080.767024,4.601955e+08,246219.340371,57.0,...,12.0,14.0,3.0,1.047713e+08,6.422148e+07,3.321578e+07,0.95,0.930,1.0,1.0
7,2025-04-01 14:00:00.000 UTC,142,111,66,60,5.869603e+08,311246.523260,4.781272e+08,253535.842057,98.0,...,21.0,17.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2025-04-01 16:00:00.000 UTC,122,103,62,55,1.321209e+08,68913.296488,1.362937e+08,71089.779751,82.0,...,13.0,11.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2025-04-01 18:00:00.000 UTC,137,98,54,55,5.251323e+08,275222.083697,5.202348e+08,272655.318074,97.0,...,16.0,25.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
DF_common_final.dtypes.rename("dtype").to_frame()

,dtype
time_bucket,str
supply_tx_count,int64
withdrawal_tx_count,int64
unique_suppliers,int64
unique_withdraw_users,int64
supply_amount_value_usd,float64
supply_amount_value_eth,float64
withdrawal_amount_value_usd,float64
withdrawal_amount_value_eth,float64
borrow_tx_count,float64


In [3]:
# additional liquidity metrics

DF_common_final["net_liquidity_flow_usd"] = DF_common_final["supply_amount_value_usd"] - DF_common_final["withdrawal_amount_value_usd"]
DF_common_final["net_liquidity_flow_eth"] = DF_common_final["supply_amount_value_eth"] - DF_common_final["withdrawal_amount_value_eth"]

# Supply/Withdrawal Ratio
DF_common_final["supply_withdrawal_ratio"] = DF_common_final["supply_amount_value_usd"] / DF_common_final["withdrawal_amount_value_usd"]
# DF_common_final["supply_withdrawal_ratio_eth"] = DF_common_final["supply_amount_value_eth"] / DF_common_final["withdrawal_amount_value_eth"]

# Liquidity Growth Rate
DF_common_final["liquidity_growth_rate"] = (
    DF_common_final["net_liquidity_flow_usd"] - DF_common_final["net_liquidity_flow_usd"].shift(1)
) / DF_common_final["net_liquidity_flow_usd"].shift(1).abs()

# Average Supply Size
DF_common_final["avg_supply_size_usd"] = DF_common_final["supply_amount_value_usd"] / DF_common_final["supply_tx_count"]

# Average Withdrawal Size
DF_common_final["avg_withdrawal_size_usd"] = DF_common_final["withdrawal_amount_value_usd"] / DF_common_final["withdrawal_tx_count"]

new_cols = [
    "net_liquidity_flow_usd",
    "net_liquidity_flow_eth",
    "supply_withdrawal_ratio",
    # "supply_withdrawal_ratio_eth",
    "liquidity_growth_rate",
    "avg_supply_size_usd",
    "avg_withdrawal_size_usd",
]

DF_common_final[["time_bucket"] + new_cols]

,time_bucket,net_liquidity_flow_usd,net_liquidity_flow_eth,supply_withdrawal_ratio,liquidity_growth_rate,avg_supply_size_usd,avg_withdrawal_size_usd
0,2025-04-01 00:00:00.000 UTC,8.382569e+07,45882.182645,1.858274,NaN,1.696200e+06,8.720335e+05
1,2025-04-01 02:00:00.000 UTC,-1.535459e+07,-8365.276605,0.942399,-1.183173,2.165622e+06,2.538731e+06
2,2025-04-01 04:00:00.000 UTC,-2.333313e+07,-12674.449452,0.824533,-0.519620,1.522833e+06,1.329771e+06
3,2025-04-01 06:00:00.000 UTC,3.532234e+06,1905.173024,1.005279,1.151383,5.214384e+06,5.818463e+06
4,2025-04-01 08:00:00.000 UTC,-1.469469e+06,-781.894221,0.996023,-1.416017,2.666972e+06,3.131455e+06
...,...,...,...,...,...,...,...
4375,2026-03-31 14:00:00.000 UTC,0.000000e+00,0.000000,NaN,NaN,0.000000e+00,0.000000e+00
4376,2026-03-31 16:00:00.000 UTC,0.000000e+00,0.000000,NaN,NaN,0.000000e+00,0.000000e+00
4377,2026-03-31 18:00:00.000 UTC,0.000000e+00,0.000000,NaN,NaN,0.000000e+00,0.000000e+00
4378,2026-03-31 20:00:00.000 UTC,0.000000e+00,0.000000,NaN,NaN,0.000000e+00,0.000000e+00


In [4]:
temp = adv.statistical_validation(DF_common_final, columns = new_cols, save=False)
display(temp[["column", "null_pct", "zero_pct", "negative_pct",]])

,column,null_pct,zero_pct,negative_pct
0,net_liquidity_flow_usd,0.000,0.274,46.2557
1,net_liquidity_flow_eth,0.000,0.274,46.2557
2,supply_withdrawal_ratio,0.274,0.000,0.0000
3,liquidity_growth_rate,0.274,0.000,51.7170
4,avg_supply_size_usd,0.000,0.274,0.0000
5,avg_withdrawal_size_usd,0.000,0.274,0.0000


In [5]:
# additional borrow metrics
#avg_collateral_base col has null values, so adjusted for it
DF_common_final["net_borrow_demand_usd"] = DF_common_final["borrow_amount_value_usd"] - DF_common_final["repay_amount_value_usd"]
DF_common_final["net_borrow_demand_eth"] = DF_common_final["borrow_amount_value_eth"] - DF_common_final["repay_amount_value_eth"]

DF_common_final["borrow_repay_ratio"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["repay_amount_value_usd"].replace(0, pd.NA)

DF_common_final["avg_borrow_size_usd"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["borrow_tx_count"].replace(0, pd.NA)
DF_common_final["avg_borrow_size_eth"] = DF_common_final["borrow_amount_value_eth"] / DF_common_final["borrow_tx_count"].replace(0, pd.NA)

DF_common_final["avg_repay_size_usd"] = DF_common_final["repay_amount_value_usd"] / DF_common_final["repay_tx_count"].replace(0, pd.NA)
DF_common_final["avg_repay_size_eth"] = DF_common_final["repay_amount_value_eth"] / DF_common_final["repay_tx_count"].replace(0, pd.NA)

DF_common_final["borrow_growth"] = (
    DF_common_final["borrow_amount_value_usd"] - DF_common_final["borrow_amount_value_usd"].shift(1)
) / DF_common_final["borrow_amount_value_usd"].shift(1).replace(0, pd.NA)

DF_common_final["debt_expansion_ratio"] = DF_common_final["avg_total_debt_base"] / DF_common_final["avg_total_collateral_base"].replace(0, pd.NA)

borrow_cols = [
    "net_borrow_demand_usd",
    "net_borrow_demand_eth",
    "borrow_repay_ratio",
    "avg_borrow_size_usd",
    "avg_borrow_size_eth",
    "avg_repay_size_usd",
    "avg_repay_size_eth",
    "borrow_growth",
    "debt_expansion_ratio",
]

DF_common_final[["time_bucket"] + borrow_cols]

,time_bucket,net_borrow_demand_usd,net_borrow_demand_eth,borrow_repay_ratio,avg_borrow_size_usd,avg_borrow_size_eth,avg_repay_size_usd,avg_repay_size_eth,borrow_growth,debt_expansion_ratio
0,2025-04-01 00:00:00.000 UTC,-1.823391e+07,-9980.372600,0.361544,191212.864321,104.660798,607646.916898,332.596929,NaN,NaN
1,2025-04-01 02:00:00.000 UTC,-9.333290e+06,-5084.850496,0.593182,234635.539624,127.831088,395554.337478,215.500924,0.317986,0.335059
2,2025-04-01 04:00:00.000 UTC,-6.644618e+06,-3609.323164,0.596064,204272.103915,110.959703,365548.416095,198.564198,-0.279509,0.344723
3,2025-04-01 06:00:00.000 UTC,-2.643288e+07,-14257.018230,0.525797,457950.540778,247.004117,831965.864331,448.735548,1.989154,NaN
4,2025-04-01 08:00:00.000 UTC,-1.825093e+07,-9711.261500,0.467345,168559.092892,89.689725,482592.097891,256.785709,-0.453642,0.377996
...,...,...,...,...,...,...,...,...,...,...
4375,2026-03-31 14:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.266605
4376,2026-03-31 16:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.708426
4377,2026-03-31 18:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.859296
4378,2026-03-31 20:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.821796


In [6]:
temp_1 = adv.statistical_validation(DF_common_final, columns = borrow_cols, save=False)
display(temp_1[["column", "null_pct", "zero_pct", "negative_pct",]])

,column,null_pct,zero_pct,negative_pct
0,net_borrow_demand_usd,0.2740,0.0000,47.6419
1,net_borrow_demand_eth,0.2740,0.0000,47.6648
2,borrow_repay_ratio,0.2740,0.0000,0.0000
3,avg_borrow_size_usd,0.2740,0.0000,0.0000
4,avg_borrow_size_eth,0.2740,0.0000,0.0000
5,avg_repay_size_usd,0.2740,0.0000,0.0000
6,avg_repay_size_eth,0.2740,0.0000,0.0000
7,borrow_growth,0.2968,0.0000,50.0114
8,debt_expansion_ratio,51.0274,6.5268,0.0000


In [7]:
#null value columns are used so expecting approximately same pct of null values
# implemented correction for NULL and zero values so there is no division by zero or such things

DF_common_final["collateralization_ratio"] = DF_common_final["avg_total_collateral_base"] / DF_common_final["avg_total_debt_base"].replace(0, pd.NA)

DF_common_final["borrow_capacity_utilization"] = DF_common_final["avg_total_debt_base"] / (
    DF_common_final["avg_total_debt_base"] + DF_common_final["avg_available_borrows_base"]
).replace(0, pd.NA)

DF_common_final["remaining_borrow_capacity"] = DF_common_final["avg_available_borrows_base"] / DF_common_final["avg_total_collateral_base"].replace(0, pd.NA)

DF_common_final["risk_buffer"] = DF_common_final["avg_current_liquidation_threshold"] - DF_common_final["avg_ltv"]

DF_common_final["ltv_utilization"] = DF_common_final["avg_ltv"] / DF_common_final["avg_current_liquidation_threshold"].replace(0, pd.NA)

DF_common_final["distance_to_liquidation"] = 1 - DF_common_final["ltv_utilization"]

risk_cols = [
    "collateralization_ratio",
    "borrow_capacity_utilization",
    "remaining_borrow_capacity",
    "risk_buffer",
    "ltv_utilization",
    "distance_to_liquidation",
]

DF_common_final[["time_bucket"] + risk_cols]

,time_bucket,collateralization_ratio,borrow_capacity_utilization,remaining_borrow_capacity,risk_buffer,ltv_utilization,distance_to_liquidation
0,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-04-01 02:00:00.000 UTC,2.984547,0.426827,0.449941,0.0250,0.969136,0.030864
2,2025-04-01 04:00:00.000 UTC,2.900878,0.439138,0.440277,0.0250,0.969136,0.030864
3,2025-04-01 06:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-04-01 08:00:00.000 UTC,2.645531,0.503995,0.372004,0.0300,0.961538,0.038462
...,...,...,...,...,...,...,...
4375,2026-03-31 14:00:00.000 UTC,3.750864,0.286672,0.663395,0.0200,0.978947,0.021053
4376,2026-03-31 16:00:00.000 UTC,1.41158,0.880032,0.096574,0.0125,0.96988,0.03012
4377,2026-03-31 18:00:00.000 UTC,1.163743,0.954774,0.040704,0.0200,0.978261,0.021739
4378,2026-03-31 20:00:00.000 UTC,1.216847,1.0,0.0,0.0496,0.93641,0.06359


In [8]:
temp_2 = adv.statistical_validation(DF_common_final, columns = risk_cols, save=False)
display(temp_2[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,collateralization_ratio,54.2237,0.0000,69.809948,24.363736,4.853533
1,borrow_capacity_utilization,51.0274,6.5268,0.650230,0.438706,0.990000
2,remaining_borrow_capacity,51.0274,1.7716,0.282379,0.854471,0.929380
3,risk_buffer,50.2055,1.6506,0.028557,0.948144,0.050000
4,ltv_utilization,51.0274,0.0932,0.963699,0.040208,0.978947
5,distance_to_liquidation,51.0274,0.0000,0.036301,1.067420,0.064103


In [9]:
#liquidation metrics
# the cols use the highest null val cols 
DF_common_final["liquidation_rate"] = DF_common_final["liquidation_tx_count"] / DF_common_final["borrow_tx_count"].replace(0, pd.NA)

# DF_common_final["liquidation_volume_ratio_usd"] = DF_common_final["liquidation_debt_covered_value_usd"] / DF_common_final["borrow_amount_value_usd"].replace(0, pd.NA)
DF_common_final["liquidation_volume_ratio_eth"] = DF_common_final["liquidation_debt_covered_value_eth"] / DF_common_final["borrow_amount_value_eth"].replace(0, pd.NA)

DF_common_final["liquidation_severity_usd"] = DF_common_final["liquidated_collateral_value_usd"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)
DF_common_final["liquidation_severity_eth"] = DF_common_final["liquidated_collateral_value_eth"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)

DF_common_final["avg_liquidation_debt_usd"] = DF_common_final["liquidation_debt_covered_value_usd"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)
DF_common_final["avg_liquidation_debt_eth"] = DF_common_final["liquidation_debt_covered_value_eth"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)

DF_common_final["liquidator_concentration"] = DF_common_final["unique_liquidators"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)

DF_common_final["liquidation_user_ratio"] = DF_common_final["unique_liquidated_users"] / DF_common_final["unique_borrowers"].replace(0, pd.NA)

liquidation_cols = [
    "liquidation_rate",
    # "liquidation_volume_ratio_usd" 
    "liquidation_volume_ratio_eth",
    "liquidation_severity_usd", "liquidation_severity_eth",
    "avg_liquidation_debt_usd", "avg_liquidation_debt_eth",
    "liquidator_concentration",
    "liquidation_user_ratio",
]

DF_common_final[["time_bucket"] + liquidation_cols]

,time_bucket,liquidation_rate,liquidation_volume_ratio_eth,liquidation_severity_usd,liquidation_severity_eth,avg_liquidation_debt_usd,avg_liquidation_debt_eth,liquidator_concentration,liquidation_user_ratio
0,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-04-01 02:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-04-01 04:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-04-01 06:00:00.000 UTC,0.062500,0.000692,5307.511474,2.862703,5071.416488,2.735351,1.0,0.093023
4,2025-04-01 08:00:00.000 UTC,0.021053,0.002476,20619.510943,10.971580,19821.802863,10.547017,1.0,0.029851
...,...,...,...,...,...,...,...,...,...
4375,2026-03-31 14:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4376,2026-03-31 16:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4377,2026-03-31 18:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4378,2026-03-31 20:00:00.000 UTC,NaN,NaN,0.000000,0.000000,0.000000,0.000000,1.0,NaN


In [10]:
temp_3 = adv.statistical_validation(DF_common_final, columns = liquidation_cols, save=False)
display(temp_3[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,liquidation_rate,72.1005,0.0000,0.125878,3.969673,0.408436
1,liquidation_volume_ratio_eth,72.1005,0.0000,0.014529,5.760878,0.049420
2,liquidation_severity_usd,72.0320,0.5714,20079.369443,5.410396,77912.459623
3,liquidation_severity_eth,72.0320,0.5714,6.835735,5.143350,27.342966
4,avg_liquidation_debt_usd,72.0320,0.2449,19054.858960,5.429671,73581.492818
5,avg_liquidation_debt_eth,72.0320,0.2449,6.486090,5.164406,25.518208
6,liquidator_concentration,72.0320,0.0000,0.858994,0.257416,1.000000
7,liquidation_user_ratio,72.1005,0.0000,0.174334,3.907807,0.589843


In [11]:
#user metrics, though sampled user are less. So not that effective as other values.

DF_common_final["borrower_participation_rate"] = DF_common_final["unique_borrowers"] / (
    DF_common_final["unique_borrowers"] + DF_common_final["unique_suppliers"]
).replace(0, pd.NA)

DF_common_final["repayment_discipline"] = DF_common_final["unique_repayers"] / DF_common_final["unique_borrowers"].replace(0, pd.NA)

DF_common_final["supplier_activity"] = DF_common_final["supply_tx_count"] / DF_common_final["unique_suppliers"].replace(0, pd.NA)

DF_common_final["borrower_activity"] = DF_common_final["borrow_tx_count"] / DF_common_final["unique_borrowers"].replace(0, pd.NA)

DF_common_final["collateral_usage_rate"] = DF_common_final["collateral_enabled_count"] / (
    DF_common_final["collateral_enabled_count"] + DF_common_final["collateral_disabled_count"]
).replace(0, pd.NA)

DF_common_final["collateral_adoption_rate"] = DF_common_final["unique_collateral_enable_users"] / DF_common_final["unique_suppliers"].replace(0, pd.NA)

user_cols= [
    "borrower_participation_rate",
    "repayment_discipline",
    "supplier_activity",
    "borrower_activity",
    "collateral_usage_rate",
    "collateral_adoption_rate",
]

DF_common_final[["time_bucket"] + user_cols]

,time_bucket,borrower_participation_rate,repayment_discipline,supplier_activity,borrower_activity,collateral_usage_rate,collateral_adoption_rate
0,2025-04-01 00:00:00.000 UTC,0.406593,0.729730,1.981481,1.459459,0.508380,1.092593
1,2025-04-01 02:00:00.000 UTC,0.439252,0.893617,1.933333,1.234043,0.573333,1.016667
2,2025-04-01 04:00:00.000 UTC,0.493333,1.027027,1.894737,1.297297,0.524194,1.078947
3,2025-04-01 06:00:00.000 UTC,0.425743,0.837209,2.224138,1.488372,0.560976,1.000000
4,2025-04-01 08:00:00.000 UTC,0.496296,0.626866,2.029412,1.417910,0.569948,1.000000
...,...,...,...,...,...,...,...
4375,2026-03-31 14:00:00.000 UTC,NaN,NaN,1.884298,NaN,0.533981,0.380165
4376,2026-03-31 16:00:00.000 UTC,NaN,NaN,2.372340,NaN,0.443396,0.329787
4377,2026-03-31 18:00:00.000 UTC,NaN,NaN,1.607843,NaN,0.598131,0.529412
4378,2026-03-31 20:00:00.000 UTC,NaN,NaN,1.486726,NaN,0.470588,0.353982


In [12]:
temp_4 = adv.statistical_validation(DF_common_final, columns = user_cols, save=False)
display(temp_4[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,borrower_participation_rate,0.274,0.0,0.429398,0.152804,0.532396
1,repayment_discipline,0.274,0.0,0.740296,0.394095,1.203841
2,supplier_activity,0.000,0.0,1.887440,0.188256,2.513692
3,borrower_activity,0.274,0.0,1.432692,0.196811,1.949872
4,collateral_usage_rate,0.000,0.0,0.542479,0.076014,0.613225
5,collateral_adoption_rate,0.000,0.0,0.735618,0.256423,1.018182


In [13]:
# values will be variably sparse , because flashloan tx count col hads 47.5% of null values
DF_common_final["avg_flashloan_size_usd"] = DF_common_final["flashloan_amount_value_usd"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)
DF_common_final["avg_flashloan_size_eth"] = DF_common_final["flashloan_amount_value_eth"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)

DF_common_final["flashloan_fee_rate"] = DF_common_final["flashloan_premium_value_usd"] / DF_common_final["flashloan_amount_value_usd"].replace(0, pd.NA)

DF_common_final["flashloan_usage_intensity"] = DF_common_final["flashloan_tx_count"] / (
    DF_common_final["borrow_tx_count"] + DF_common_final["supply_tx_count"]
).replace(0, pd.NA)

DF_common_final["flashloan_user_activity"] = DF_common_final["flashloan_tx_count"] / DF_common_final["unique_flashloan_initiators"].replace(0, pd.NA)

DF_common_final["variable_debt_flashloan_ratio"] = DF_common_final["variable_flashloan_tx_count"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)  # ⚠️ 47.5% nulls

DF_common_final["no_debt_flashloan_ratio"] = DF_common_final["no_open_debt_flashloan_tx_count"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)

flashloand_cols = [
    "avg_flashloan_size_usd", "avg_flashloan_size_eth",
    "flashloan_fee_rate",
    "flashloan_usage_intensity",
    "flashloan_user_activity",
    "variable_debt_flashloan_ratio",
    "no_debt_flashloan_ratio",
]

DF_common_final[["time_bucket"] + flashloand_cols]

,time_bucket,avg_flashloan_size_usd,avg_flashloan_size_eth,flashloan_fee_rate,flashloan_usage_intensity,flashloan_user_activity,variable_debt_flashloan_ratio,no_debt_flashloan_ratio
0,2025-04-01 00:00:00.000 UTC,36720.841001,20.099236,0.00026,0.161491,1.300000,0.269231,0.730769
1,2025-04-01 02:00:00.000 UTC,284037.728669,154.745585,0.000465,0.132184,1.210526,0.173913,0.826087
2,2025-04-01 04:00:00.000 UTC,28944.448637,15.722497,0.000032,0.075000,1.285714,0.111111,0.888889
3,2025-04-01 06:00:00.000 UTC,66836.817851,36.049638,0.00013,0.088083,2.428571,0.647059,0.352941
4,2025-04-01 08:00:00.000 UTC,76216.654241,40.554512,0.000332,0.120172,1.647059,0.392857,0.607143
...,...,...,...,...,...,...,...,...
4375,2026-03-31 14:00:00.000 UTC,0.000000,0.000000,<NA>,NaN,2.444444,0.000000,1.000000
4376,2026-03-31 16:00:00.000 UTC,0.000000,0.000000,<NA>,NaN,2.142857,0.000000,1.000000
4377,2026-03-31 18:00:00.000 UTC,0.000000,0.000000,<NA>,NaN,2.166667,0.076923,0.923077
4378,2026-03-31 20:00:00.000 UTC,0.000000,0.000000,<NA>,NaN,2.428571,0.000000,1.000000


In [14]:
temp_5 = adv.statistical_validation(DF_common_final, columns = flashloand_cols, save=False)
display(temp_5[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,avg_flashloan_size_usd,0.1598,0.2744,119613.791718,5.175542,351631.217641
1,avg_flashloan_size_eth,0.1598,0.2744,39.453525,4.505349,110.611226
2,flashloan_fee_rate,0.4338,0.8255,0.000294,0.599335,0.000500
3,flashloan_usage_intensity,0.4338,0.0000,0.070549,0.541116,0.140962
4,flashloan_user_activity,0.1598,0.0000,1.877877,0.553289,3.833333
5,variable_debt_flashloan_ratio,0.1598,29.7279,0.250996,0.882580,0.625000
6,no_debt_flashloan_ratio,0.1598,0.2744,0.749004,0.295758,1.000000


In [15]:
tx_cols = [
    "supply_tx_count", "withdrawal_tx_count", "borrow_tx_count",
    "repay_tx_count", "flashloan_tx_count", "liquidation_tx_count"
]

DF_common_final["total_activity"] = DF_common_final[tx_cols].fillna(0).sum(axis=1)

user_cols = ["unique_suppliers", "unique_borrowers", "unique_flashloan_initiators"]
DF_common_final["user_activity"] = DF_common_final[user_cols].fillna(0).sum(axis=1)

turnover_usd_cols = ["supply_amount_value_usd", "withdrawal_amount_value_usd", "borrow_amount_value_usd", "repay_amount_value_usd"]
turnover_eth_cols = ["supply_amount_value_eth", "withdrawal_amount_value_eth", "borrow_amount_value_eth", "repay_amount_value_eth"]
DF_common_final["protocol_turnover_usd"] = DF_common_final[turnover_usd_cols].fillna(0).sum(axis=1)
DF_common_final["protocol_turnover_eth"] = DF_common_final[turnover_eth_cols].fillna(0).sum(axis=1)

DF_common_final["leverage_indicator"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["supply_amount_value_usd"].replace(0, pd.NA)

DF_common_final["market_stress_index_usd"] = (
    DF_common_final["liquidation_tx_count"] *
    DF_common_final["liquidation_debt_covered_value_usd"]
) / DF_common_final["borrow_amount_value_usd"].replace(0, pd.NA)  # ⚠️ 47% nulls
DF_common_final["market_stress_index_eth"] = (
    DF_common_final["liquidation_tx_count"] *
    DF_common_final["liquidation_debt_covered_value_eth"]
) / DF_common_final["borrow_amount_value_eth"].replace(0, pd.NA)  # ⚠️ 47% nulls

DF_common_final["capital_efficiency"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["avg_total_collateral_base"].replace(0, pd.NA)  # ⚠️ 21% nulls

market_cols = [
    "total_activity",
    "user_activity",
    "protocol_turnover_usd", "protocol_turnover_eth",
    "leverage_indicator",
    "market_stress_index_usd", "market_stress_index_eth",
    "capital_efficiency",
]

DF_common_final[["time_bucket"] + market_cols]

,time_bucket,total_activity,user_activity,protocol_turnover_usd,protocol_turnover_eth,leverage_indicator,market_stress_index_usd,market_stress_index_eth,capital_efficiency
0,2025-04-01 00:00:00.000 UTC,346.0,111.0,3.180461e+08,174083.255979,0.056892,NaN,NaN,NaN
1,2025-04-01 02:00:00.000 UTC,360.0,126.0,5.543299e+08,302003.055537,0.054173,NaN,NaN,0.091242
2,2025-04-01 04:00:00.000 UTC,274.0,82.0,2.688759e+08,146051.770761,0.089426,NaN,NaN,0.065093
3,2025-04-01 06:00:00.000 UTC,396.0,108.0,1.426829e+09,769585.811424,0.043572,0.002769,0.002769,NaN
4,2025-04-01 08:00:00.000 UTC,452.0,152.0,7.878310e+08,419201.167644,0.043509,0.004951,0.004951,12683.785135
...,...,...,...,...,...,...,...,...,...
4375,2026-03-31 14:00:00.000 UTC,407.0,130.0,0.000000e+00,0.000000,<NA>,NaN,NaN,NaN
4376,2026-03-31 16:00:00.000 UTC,431.0,101.0,0.000000e+00,0.000000,<NA>,NaN,NaN,NaN
4377,2026-03-31 18:00:00.000 UTC,332.0,108.0,0.000000e+00,0.000000,<NA>,NaN,NaN,NaN
4378,2026-03-31 20:00:00.000 UTC,346.0,120.0,0.000000e+00,0.000000,<NA>,NaN,NaN,NaN


In [16]:
temp_6 = adv.statistical_validation(DF_common_final, columns = market_cols, save=False)
display(temp_6[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,total_activity,0.0000,0.000,5.628872e+02,0.606534,1.074200e+03
1,user_activity,0.0000,0.000,1.823893e+02,0.427345,3.170000e+02
2,protocol_turnover_usd,0.0000,0.274,4.959540e+08,1.658548,1.438656e+09
3,protocol_turnover_eth,0.0000,0.274,1.746974e+05,1.349200,5.560819e+05
4,leverage_indicator,0.2740,0.000,3.922095e-01,0.959325,9.608501e-01
5,market_stress_index_usd,72.1005,0.000,6.770309e+00,11.030712,1.372853e+00
6,market_stress_index_eth,72.1005,0.000,6.770959e+00,11.031134,1.372854e+00
7,capital_efficiency,51.1872,0.000,2.877547e+12,35.124298,1.424612e+05


In [17]:
print(f" {DF_common_1.shape[0]} rows x {DF_common_1.shape[1]} cols")
display(DF_common_1.head(PREVIEW_ROWS))

 54198 rows x 9 cols


,time_bucket,asset,asset_symbol,last_borrow_rate,liquidity_rate,variable_borrow_rate,liquidity_index,variable_borrow_index,update_count_liquidity
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,0.005103,0.000260,0.005101,1.003301,1.022170,21
1,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0.045000,0.000000,0.045000,1.000000,1.123769,5
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,0.033222,0.011037,0.033222,1.038794,1.074808,5
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0.048095,0.026601,0.048095,1.121824,1.179228,2
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,0.004968,0.001172,0.004968,1.001093,1.009286,6
5,2025-04-01 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0.046031,0.029176,0.045980,1.125029,1.164874,91
6,2025-04-01 00:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,0.089417,0.017815,0.088239,1.023239,1.127670,3
7,2025-04-01 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0.026735,0.020251,0.026735,1.042603,1.067234,96
8,2025-04-01 00:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,0.048260,0.032141,0.048260,1.119375,1.163539,73
9,2025-04-01 00:00:00.000 UTC,0xf939e0a03fb07f59a73314e73794be0e57ac1b4e,crvUSD,NaN,0.035663,0.054515,1.144648,1.189138,1


In [18]:
DF_common_1.dtypes.rename("dtype").to_frame()

,dtype
time_bucket,str
asset,str
asset_symbol,str
last_borrow_rate,float64
liquidity_rate,float64
variable_borrow_rate,float64
liquidity_index,float64
variable_borrow_index,float64
update_count_liquidity,int64


In [19]:
#Rate dynamics, many cols have null values, adjusting for that.
DF_common_1["borrow_supply_spread"] = DF_common_1["last_borrow_rate"] - DF_common_1["liquidity_rate"]  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["variable_supply_spread"] = DF_common_1["variable_borrow_rate"] - DF_common_1["liquidity_rate"]

DF_common_1["borrow_rate_premium"] = DF_common_1["last_borrow_rate"] / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 11.6% nulls + 5.9% zeros guarded

DF_common_1["variable_rate_premium"] = DF_common_1["variable_borrow_rate"] / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

# DF_common_1["rate_change"] = DF_common_1["last_borrow_rate"] - DF_common_1["last_borrow_rate"].shift(1)  # ⚠️ 11.6% nulls
DF_common_1["rate_change"] = (DF_common_1["last_borrow_rate"] - DF_common_1["last_borrow_rate"].shift(1)) / DF_common_1["last_borrow_rate"].replace(0, pd.NA)

# DF_common_1["liquidity_rate_change"] = DF_common_1["liquidity_rate"] - DF_common_1["liquidity_rate"].shift(1)
DF_common_1["liquidity_rate_change"] = (DF_common_1["liquidity_rate"] - DF_common_1["liquidity_rate"].shift(1)) / DF_common_1["liquidity_rate"].replace(0, pd.NA)

DF_common_1["variable_borrow_rate_change"] = DF_common_1["variable_borrow_rate"] - DF_common_1["variable_borrow_rate"].shift(1)

rate_dynamics_cols = [
    "borrow_supply_spread",
    "variable_supply_spread",
    "borrow_rate_premium",
    "variable_rate_premium",
    "rate_change",
    "liquidity_rate_change",
    "variable_borrow_rate_change",
]

DF_common_1[["time_bucket"] + rate_dynamics_cols]

,time_bucket,borrow_supply_spread,variable_supply_spread,borrow_rate_premium,variable_rate_premium,rate_change,liquidity_rate_change,variable_borrow_rate_change
0,2025-04-01 00:00:00.000 UTC,0.004843,0.004841,19.607808,19.602141,NaN,NaN,NaN
1,2025-04-01 00:00:00.000 UTC,0.045000,0.045000,<NA>,<NA>,0.886601,<NA>,0.039899
2,2025-04-01 00:00:00.000 UTC,0.022185,0.022185,3.010077,3.010076,-0.354533,1.0,-0.011778
3,2025-04-01 00:00:00.000 UTC,0.021494,0.021494,1.808023,1.808023,0.309244,0.585093,0.014873
4,2025-04-01 00:00:00.000 UTC,0.003795,0.003795,4.23801,4.238025,-8.681822,-21.694209,-0.043127
...,...,...,...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,0.009802,0.009801,1.394598,1.394564,NaN,0.579001,-0.001623
54194,2026-03-30 22:00:00.000 UTC,0.005737,0.005737,1.345744,1.345743,-0.551345,-0.497,-0.012311
54195,2026-03-30 22:00:00.000 UTC,0.014329,0.014329,2.809029,2.809029,-0.003637,-1.094935,-0.000081
54196,2026-03-30 22:00:00.000 UTC,0.003035,0.003035,181.34336,181.342655,-6.291501,-469.719637,-0.019198


In [20]:
temp_7 = adv.statistical_validation(DF_common_1, columns = rate_dynamics_cols, save=False)
display(temp_7[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,borrow_supply_spread,16.4157,0.0000,1.847150e-02,1.695964,0.055000
1,variable_supply_spread,0.0000,0.0018,1.752195e-02,0.942400,0.055000
2,borrow_rate_premium,23.1503,0.0000,3.671160e+07,124.580208,83.177876
3,variable_rate_premium,7.5556,0.0000,1.559854e+03,57.999772,94.106102
4,rate_change,30.4255,0.0000,-2.494077e+02,54.879207,0.963385
5,liquidity_rate_change,7.5575,0.0000,-1.505631e+09,137.444463,1.000000
6,variable_borrow_rate_change,0.0018,0.0000,4.736688e-07,85192.392449,0.054838


In [21]:
# growth metrics — per-asset shift(1) (DF_common_1 is asset-level, so t-1 is the same asset's prev 6h bucket)
# index cols are >= 1.0 (no zeros / nulls), so no denominator guard needed
DF_common_1["liquidity_index_growth"] = (
    DF_common_1["liquidity_index"] - DF_common_1.groupby("asset")["liquidity_index"].shift(1)
) / DF_common_1.groupby("asset")["liquidity_index"].shift(1)

DF_common_1["variable_borrow_index_growth"] = (
    DF_common_1["variable_borrow_index"] - DF_common_1.groupby("asset")["variable_borrow_index"].shift(1)
) / DF_common_1.groupby("asset")["variable_borrow_index"].shift(1)

# interest / debt accrual velocity = per-asset delta of the index
DF_common_1["interest_accrual_velocity"] = DF_common_1["liquidity_index"] - DF_common_1.groupby("asset")["liquidity_index"].shift(1)
DF_common_1["debt_accrual_velocity"] = DF_common_1["variable_borrow_index"] - DF_common_1.groupby("asset")["variable_borrow_index"].shift(1)

growth_cols = [
    "liquidity_index_growth",
    "variable_borrow_index_growth",
    "interest_accrual_velocity",
    "debt_accrual_velocity",
]

DF_common_1[["time_bucket"] + growth_cols]

,time_bucket,liquidity_index_growth,variable_borrow_index_growth,interest_accrual_velocity,debt_accrual_velocity
0,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
1,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
2,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
3,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
4,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,5.398111e-06,7.696396e-06,6.290285e-06,9.411540e-06
54194,2026-03-30 22:00:00.000 UTC,3.778214e-06,5.082917e-06,4.012313e-06,5.563039e-06
54195,2026-03-30 22:00:00.000 UTC,1.748105e-06,4.910503e-06,1.772118e-06,5.065598e-06
54196,2026-03-30 22:00:00.000 UTC,3.483201e-09,6.316533e-07,3.483878e-09,6.341862e-07


In [22]:
temp_8 = adv.statistical_validation(DF_common_1, columns = growth_cols, save=False)
display(temp_8[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,liquidity_index_growth,0.072,7.5758,0.000008,20.703815,0.000017
1,variable_borrow_index_growth,0.072,0.0018,0.000020,12.333003,0.000045
2,interest_accrual_velocity,0.072,7.5758,0.000009,21.568366,0.000018
3,debt_accrual_velocity,0.072,0.0018,0.000023,12.876553,0.000051


In [23]:
# rate risk metrics — same-unit ratios (no usd/eth split)
# liquidity_rate 5.9% zeros guarded, last_borrow_rate 11.6% nulls
DF_common_1["borrow_pressure"] = DF_common_1["variable_borrow_rate"] / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

DF_common_1["lending_attractiveness"] = DF_common_1["liquidity_rate"] / DF_common_1["last_borrow_rate"].replace(0, pd.NA)  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["reserve_stress_score"] = (
    DF_common_1["variable_borrow_rate"] - DF_common_1["liquidity_rate"]
) / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

rate_risk_cols = [
    "borrow_pressure",
    "lending_attractiveness",
    "reserve_stress_score",
]

DF_common_1[["time_bucket"] + rate_risk_cols]

,time_bucket,borrow_pressure,lending_attractiveness,reserve_stress_score
0,2025-04-01 00:00:00.000 UTC,19.602141,0.051000,18.602141
1,2025-04-01 00:00:00.000 UTC,<NA>,0.000000,<NA>
2,2025-04-01 00:00:00.000 UTC,3.010076,0.332217,2.010076
3,2025-04-01 00:00:00.000 UTC,1.808023,0.553090,0.808023
4,2025-04-01 00:00:00.000 UTC,4.238025,0.235960,3.238025
...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,1.394564,0.717052,0.394564
54194,2026-03-30 22:00:00.000 UTC,1.345743,0.743083,0.345743
54195,2026-03-30 22:00:00.000 UTC,2.809029,0.355995,1.809029
54196,2026-03-30 22:00:00.000 UTC,181.342655,0.005514,180.342655


In [24]:
temp_9 = adv.statistical_validation(DF_common_1, columns = rate_risk_cols, save=False)
display(temp_9[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,borrow_pressure,7.5556,0.0000,1559.853947,57.999772,94.106102
1,lending_attractiveness,16.4157,8.0572,0.376185,0.803611,0.776771
2,reserve_stress_score,7.5556,0.0000,1558.853947,58.036979,93.106102


In [25]:
# activity / volatility metrics — rolling std per asset over 4 × 6h (24h) windows
# update_count is already a metric, so not re-created here
DF_common_1["rate_volatility"] = DF_common_1.groupby("asset")["last_borrow_rate"].transform(lambda s: s.rolling(4, min_periods=2).std())  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["liquidity_volatility"] = DF_common_1.groupby("asset")["liquidity_rate"].transform(lambda s: s.rolling(4, min_periods=2).std())

DF_common_1["borrow_volatility"] = DF_common_1.groupby("asset")["variable_borrow_rate"].transform(lambda s: s.rolling(4, min_periods=2).std())

volatility_cols = [
    "rate_volatility",
    "liquidity_volatility",
    "borrow_volatility",
]

DF_common_1[["time_bucket"] + volatility_cols]

,time_bucket,rate_volatility,liquidity_volatility,borrow_volatility
0,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
1,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
2,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
3,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
4,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN
...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,5.245031e-04,7.409617e-04,5.246930e-04
54194,2026-03-30 22:00:00.000 UTC,1.047773e-05,1.460308e-05,9.825413e-06
54195,2026-03-30 22:00:00.000 UTC,5.177098e-05,4.262686e-05,5.973130e-05
54196,2026-03-30 22:00:00.000 UTC,5.263477e-08,4.418101e-09,1.226323e-07


In [26]:
temp_10 = adv.statistical_validation(DF_common_1, columns = volatility_cols, save=False)
display(temp_10[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,rate_volatility,5.705,7.7506,0.003056,9.348940,0.003116
1,liquidity_volatility,0.072,7.5611,0.000976,8.351547,0.002005
2,borrow_volatility,0.072,7.4337,0.001248,8.646683,0.002402


In [27]:
# momentum (ML) features — per-asset rate(t) / rate(t-k); k in {1, 4} (4 × 6h = 24h), denominators guarded
DF_common_1["momentum"] = DF_common_1["last_borrow_rate"] / DF_common_1.groupby("asset")["last_borrow_rate"].shift(1).replace(0, pd.NA)  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["borrow_momentum_24h"] = DF_common_1["last_borrow_rate"] / DF_common_1.groupby("asset")["last_borrow_rate"].shift(4).replace(0, pd.NA)  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["liquidity_momentum_24h"] = DF_common_1["liquidity_rate"] / DF_common_1.groupby("asset")["liquidity_rate"].shift(4).replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

DF_common_1["index_momentum_24h"] = DF_common_1["liquidity_index"] / DF_common_1.groupby("asset")["liquidity_index"].shift(4)  # index >= 1.0, no zero-guard needed

momentum_cols = [
    "momentum",
    "borrow_momentum_24h",
    "liquidity_momentum_24h",
    "index_momentum_24h",
]

DF_common_1[["time_bucket"] + momentum_cols]

,time_bucket,momentum,borrow_momentum_24h,liquidity_momentum_24h,index_momentum_24h
0,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
1,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
2,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
3,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
4,2025-04-01 00:00:00.000 UTC,NaN,NaN,NaN,NaN
...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,1.028276,1.040468,1.082638,1.000021
54194,2026-03-30 22:00:00.000 UTC,0.999324,0.998184,0.996379,1.000015
54195,2026-03-30 22:00:00.000 UTC,0.995356,0.995465,0.991697,1.000009
54196,2026-03-30 22:00:00.000 UTC,1.000015,0.999959,0.999731,1.000000


In [28]:
temp_11 = adv.statistical_validation(DF_common_1, columns = momentum_cols, save=False)
display(temp_11[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,momentum,27.0803,0.0,1.040278e+00,1.557873,1.028077
1,borrow_momentum_24h,27.4863,0.0,1.073158e+00,5.058654,1.073163
2,liquidity_momentum_24h,7.8305,0.0,7.032880e+06,222.517242,1.181667
3,index_momentum_24h,0.2823,0.0,1.000031e+00,0.000367,1.000065


In [29]:
# cross-asset features — rank within each time_bucket + asset share of updates (asset_symbol present)
DF_common_1["rank_by_borrow_rate"] = DF_common_1.groupby("time_bucket")["last_borrow_rate"].rank()  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["rank_by_liquidity_rate"] = DF_common_1.groupby("time_bucket")["liquidity_rate"].rank()

DF_common_1["asset_share_of_updates"] = DF_common_1["update_count_liquidity"] / DF_common_1.groupby("time_bucket")["update_count_liquidity"].transform("sum").replace(0, pd.NA)

cross_asset_cols = [
    "rank_by_borrow_rate",
    "rank_by_liquidity_rate",
    "asset_share_of_updates",
]

DF_common_1[["time_bucket"] + cross_asset_cols]

,time_bucket,rank_by_borrow_rate,rank_by_liquidity_rate,asset_share_of_updates
0,2025-04-01 00:00:00.000 UTC,2.0,2.0,0.069307
1,2025-04-01 00:00:00.000 UTC,5.0,1.0,0.016502
2,2025-04-01 00:00:00.000 UTC,4.0,4.0,0.016502
3,2025-04-01 00:00:00.000 UTC,7.0,7.0,0.006601
4,2025-04-01 00:00:00.000 UTC,1.0,3.0,0.019802
...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,8.0,12.0,0.356250
54194,2026-03-30 22:00:00.000 UTC,5.0,9.0,0.225000
54195,2026-03-30 22:00:00.000 UTC,4.0,4.0,0.006250
54196,2026-03-30 22:00:00.000 UTC,1.0,1.0,0.018750


In [30]:
temp_12 = adv.statistical_validation(DF_common_1, columns = cross_asset_cols, save=False)
display(temp_12[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,rank_by_borrow_rate,16.4157,0.0,5.913313,0.566461,12.000000
1,rank_by_liquidity_rate,0.0000,0.0,6.926510,0.568201,14.000000
2,asset_share_of_updates,0.0000,0.0,0.080593,1.363972,0.306846


In [31]:
DF_common_1.dtypes.rename("dtype").to_frame()

,dtype
time_bucket,str
asset,str
asset_symbol,str
last_borrow_rate,float64
liquidity_rate,float64
variable_borrow_rate,float64
liquidity_index,float64
variable_borrow_index,float64
update_count_liquidity,int64
borrow_supply_spread,float64


In [32]:
from pathlib import Path

OUT_DIR = Path("transformed_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DF_common_2 = DF_common_1
DF_common_final_1 = DF_common_final

DF_common_final_1.to_csv(OUT_DIR / "DF_common_final_1.csv", index=False)
DF_common_2.to_csv(OUT_DIR / "DF_common_2.csv", index=False)

print(f"wrote {len(DF_common_final_1)} + {len(DF_common_2)} rows to {OUT_DIR}/")

wrote 4380 + 54198 rows to transformed_data/
